In [10]:
import numpy as np                    
import matplotlib.pyplot as plt        
from sklearn.linear_model import Ridge

In [11]:
N = 50 #number of virtual nodes
theta = 10**-10 #interval length 
p = 1.11 #normalised injection current
eta = 0.08 #input strength 
alpha = 3 #linewidth enhancemenet factor
phi = 0.0 #phase shift
kappa = 10**8
dt = 10**-12


T = N*theta # mask period
tau = 1.01*T # feedback delay time, why are these different
Nd = int(round(T / dt))  
tap_stride = int(round(theta / dt)) 

In [12]:
def generate_mask(N, rng=None):
    rng = np.random.default_rng(rng)
    return rng.random(N)   # default values between 0 and 1 range.

mask = generate_mask(N, rng=42)

In [13]:
x = 0.0
buffer = np.zeros(Nd) #it stores past Nd state of the node (keeps track of)
buffer_idx = 0 # index pointer for buffer

In [14]:
def ODEs(E, n, E_delay, v_slot, alpha, kappa, phi, p, eta):
  
  
    I = abs(E)**2  

    dEdt = (1.0 + 1j*alpha) * E - I * E + kappa * E_delay * np.exp(1j*phi)

    dndt = p + eta * v_slot - (1.0 + I) * n

    return dEdt, dndt


In [15]:
def rk4_step(E, n, E_delay, v_slot, dt, alpha, kappa, phi, p, eta):
    
   # One RK4 step. v should be v over tap_stride duration. 
 
    # k1
    k1_E, k1_n = ODEs(E, n, E_delay, v_slot, alpha, kappa, phi, p, eta)

    # k2
    E2 = E + 0.5 * dt * k1_E
    n2 = n + 0.5 * dt * k1_n
    k2_E, k2_n = ODEs(E2, n2, E_delay, v_slot, alpha, kappa, phi, p, eta)

    # k3
    E3 = E + 0.5 * dt * k2_E
    n3 = n + 0.5 * dt * k2_n
    k3_E, k3_n = ODEs(E3, n3, E_delay, v_slot, alpha, kappa, phi, p, eta)

    # k4
    E4 = E + dt * k3_E
    n4 = n + dt * k3_n
    k4_E, k4_n = ODEs(E4, n4, E_delay, v_slot, alpha, kappa, phi, p, eta)

    # combine
    E_next = E + (dt/6.0) * (k1_E + 2*k2_E + 2*k3_E + k4_E)
    n_next = n + (dt/6.0) * (k1_n + 2*k2_n + 2*k3_n + k4_n)

    return E_next, n_next